# AAVAIL Capstone — End-to-End Notebook Solution

Notebook này cung cấp lời giải triển khai cho bài AI Workflow Capstone: data ingestion từ JSON, EDA, feature engineering, dự báo doanh thu 30 ngày, so sánh baseline và nhiều mô hình, lưu model, monitoring, cùng skeleton Flask API, tests và Docker.

**Trước khi chạy**
- Notebook không kèm dữ liệu gốc AAVAIL. Tải dữ liệu từ khóa học/repository và sửa `DATA_DIR`.
- Kiểm tra schema JSON và sửa `COLUMN_MAP` theo tên cột thật.
- Target là tổng doanh thu 30 ngày sau ngày dự báo; không dùng dữ liệu tương lai làm feature.
- Chỉ số sẽ được tính từ dữ liệu thực khi chạy, không có metric giả.
- API skeleton cần hoàn thiện hàm tạo feature để khớp với huấn luyện trước khi triển khai thật.

## 1. Cài đặt thư viện

In [ ]:
# Nếu cần cài thư viện, bỏ dấu # và chạy:
# %pip install pandas numpy matplotlib scikit-learn joblib flask pytest

## 2. Import và cấu hình

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# SỬA đường dẫn này tới thư mục JSON AAVAIL thực tế.
DATA_DIR = Path("../ai-workflow-capstone/cs-train")
PROJECT_DIR = Path("aavail_capstone_output")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
REPORT_DIR = PROJECT_DIR / "reports"
FIGURE_DIR = REPORT_DIR / "figures"

for folder in [PROCESSED_DIR, ARTIFACT_DIR, REPORT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("DATA_DIR exists:", DATA_DIR.exists())
print("Resolved path:", DATA_DIR.resolve())

## 3. Data ingestion — tìm và đọc JSON

In [ ]:
def records_from_json(obj):
    if isinstance(obj, list):
        return [x for x in obj if isinstance(x, dict)]
    if isinstance(obj, dict):
        for key in ("data", "records", "items", "transactions", "results"):
            value = obj.get(key)
            if isinstance(value, list) and all(isinstance(x, dict) for x in value):
                return value
        # A single flat record
        if any(not isinstance(v, (dict, list)) for v in obj.values()):
            return [obj]
        return [pd.json_normalize(obj, sep="_").iloc[0].to_dict()]
    return []

def load_json_directory(directory):
    files = sorted(Path(directory).rglob("*.json"))
    if not files:
        raise FileNotFoundError(
            f"No JSON files under {Path(directory).resolve()}. Update DATA_DIR."
        )
    frames, errors = [], []
    for file in files:
        try:
            with file.open("r", encoding="utf-8") as f:
                obj = json.load(f)
            records = records_from_json(obj)
            if records:
                frame = pd.json_normalize(records, sep="_")
                frame["source_file"] = file.name
                frames.append(frame)
        except Exception as exc:
            errors.append((str(file), str(exc)))
    if not frames:
        raise ValueError("JSON files found, but no tabular records were extracted.")
    data = pd.concat(frames, ignore_index=True, sort=False)
    print(f"Files: {len(files)} | Records: {len(data):,} | Read errors: {len(errors)}")
    if errors[:5]:
        print("First errors:", errors[:5])
    return data

raw = load_json_directory(DATA_DIR)
display(raw.head())
print("Shape:", raw.shape)

## 4. Kiểm tra schema và xác nhận cột

In [ ]:
schema = pd.DataFrame({
    "column": raw.columns,
    "dtype": [str(raw[c].dtype) for c in raw.columns],
    "non_null": [int(raw[c].notna().sum()) for c in raw.columns],
    "examples": [raw[c].dropna().astype(str).head(3).tolist() for c in raw.columns],
})
display(schema)

In [ ]:
def find_column(columns, aliases):
    lower = {str(c).lower(): c for c in columns}
    for alias in aliases:
        if alias.lower() in lower:
            return lower[alias.lower()]
    for c in columns:
        if any(alias.lower() in str(c).lower() for alias in aliases):
            return c
    return None

ALIASES = {
    "date": ["transaction_date", "purchase_date", "timestamp", "datetime", "date", "created_at", "time"],
    "country": ["country", "country_name", "billing_country", "customer_country", "market"],
    "revenue": ["revenue", "amount", "total_price", "price", "purchase_amount", "sales"],
    "views": ["stream_views", "views", "total_views", "view_count"],
    "purchases": ["purchases", "purchase_count", "transactions", "transaction_count", "quantity"],
    "customer_id": ["customer_id", "user_id", "client_id"],
    "stream_id": ["stream_id", "stream_ids", "content_id"],
}
detected = {k: find_column(raw.columns, v) for k, v in ALIASES.items()}
print("Detected mapping:", detected)

# IMPORTANT: verify these against the schema above and edit if necessary.
COLUMN_MAP = {
    "date": detected["date"],
    "country": detected["country"],
    "revenue": detected["revenue"],
    "views": detected["views"],
    "purchases": detected["purchases"],
    "customer_id": detected["customer_id"],
    "stream_id": detected["stream_id"],
}

## 5. Chuẩn hóa dữ liệu giao dịch

In [ ]:
def normalize_transactions(frame, mapping):
    required = ["date", "country", "revenue"]
    missing = [k for k in required if not mapping.get(k) or mapping[k] not in frame.columns]
    if missing:
        raise ValueError(
            f"Required mappings missing: {missing}. Edit COLUMN_MAP after reviewing schema."
        )

    out = pd.DataFrame(index=frame.index)
    out["date"] = pd.to_datetime(frame[mapping["date"]], errors="coerce", utc=True).dt.tz_convert(None)
    out["country"] = frame[mapping["country"]].astype("string").str.strip()
    out["revenue"] = pd.to_numeric(frame[mapping["revenue"]], errors="coerce")

    for key in ("views", "purchases"):
        col = mapping.get(key)
        if col and col in frame.columns:
            out[key] = pd.to_numeric(frame[col], errors="coerce")
    for key in ("customer_id", "stream_id"):
        col = mapping.get(key)
        if col and col in frame.columns:
            out[key] = frame[col].astype("string")

    before = len(out)
    out = out.dropna(subset=["date", "country", "revenue"])
    out = out[out["country"].str.len() > 0].sort_values("date").reset_index(drop=True)
    print(f"Rows before cleaning: {before:,}; after cleaning: {len(out):,}; removed: {before-len(out):,}")
    return out

transactions = normalize_transactions(raw, COLUMN_MAP)
display(transactions.head())
transactions.to_csv(PROCESSED_DIR / "transactions_clean.csv", index=False)

## 6. Tổng hợp theo ngày và quốc gia

In [ ]:
agg = {"revenue": ("revenue", "sum")}
if "views" in transactions.columns:
    agg["stream_views"] = ("views", "sum")
if "purchases" in transactions.columns:
    agg["purchases"] = ("purchases", "sum")
if "customer_id" in transactions.columns:
    agg["unique_customers"] = ("customer_id", "nunique")
if "stream_id" in transactions.columns:
    agg["unique_streams"] = ("stream_id", "nunique")

daily = (
    transactions.groupby([pd.Grouper(key="date", freq="D"), "country"])
    .agg(**agg).reset_index().sort_values(["country", "date"])
)
display(daily.head())
print("Daily rows:", len(daily), "| Countries:", daily["country"].nunique())
daily.to_csv(PROCESSED_DIR / "daily_country_revenue.csv", index=False)

## 7. EDA — biểu đồ và kiểm tra dữ liệu

In [ ]:
print("Date range:", daily["date"].min(), "to", daily["date"].max())
print("Missing values:\n", daily.isna().sum())
display(daily.groupby("country")["revenue"].describe())

plt.figure(figsize=(12, 5))
for country, group in daily.groupby("country"):
    plt.plot(group["date"], group["revenue"], alpha=0.7, label=str(country))
plt.title("Daily revenue by country")
plt.xlabel("Date"); plt.ylabel("Daily revenue")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "daily_revenue_by_country.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(daily["revenue"].dropna(), bins=40)
plt.title("Distribution of daily revenue")
plt.xlabel("Daily revenue"); plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "revenue_distribution.png", dpi=150)
plt.show()

country_revenue = daily.groupby("country", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)
display(country_revenue)
plt.figure(figsize=(10, 5))
plt.bar(country_revenue["country"].astype(str), country_revenue["revenue"])
plt.xticks(rotation=45, ha="right")
plt.title("Total revenue by country")
plt.xlabel("Country"); plt.ylabel("Total revenue")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "total_revenue_by_country.png", dpi=150)
plt.show()

## 8. Feature engineering — dự báo tổng doanh thu 30 ngày tiếp theo

In [ ]:
HORIZON_DAYS = 30
LAGS = [1, 7, 14, 30]
ROLLING_WINDOWS = [7, 30]

def build_forecasting_table(daily_df, horizon=30):
    pieces = []
    for country, group in daily_df.groupby("country"):
        g = group.sort_values("date").set_index("date").asfreq("D")
        g["country"] = country

        # Assumption: a missing calendar date means no transactions.
        # Verify source completeness before accepting this assumption.
        g["revenue"] = g["revenue"].fillna(0)
        for col in ("stream_views", "purchases"):
            if col in g.columns:
                g[col] = g[col].fillna(0)

        # Sum revenue on the next horizon days; current date is excluded.
        g["target_revenue_next_30d"] = (
            g["revenue"].shift(-1).rolling(horizon).sum().shift(-(horizon - 1))
        )

        for lag in LAGS:
            g[f"revenue_lag_{lag}"] = g["revenue"].shift(lag)
        for window in ROLLING_WINDOWS:
            g[f"revenue_mean_{window}d"] = g["revenue"].shift(1).rolling(window).mean()
            g[f"revenue_std_{window}d"] = g["revenue"].shift(1).rolling(window).std()

        g["day_of_week"] = g.index.dayofweek
        g["month"] = g.index.month
        g["day_of_year"] = g.index.dayofyear

        if "stream_views" in g.columns:
            g["views_lag_1"] = g["stream_views"].shift(1)
            g["views_mean_7d"] = g["stream_views"].shift(1).rolling(7).mean()
        if "purchases" in g.columns:
            g["purchases_lag_1"] = g["purchases"].shift(1)
            g["purchases_mean_7d"] = g["purchases"].shift(1).rolling(7).mean()

        pieces.append(g.reset_index(names="date"))
    result = pd.concat(pieces, ignore_index=True).replace([np.inf, -np.inf], np.nan)
    return result

forecast_df = build_forecasting_table(daily, HORIZON_DAYS)
display(forecast_df.head())
print("Forecast table shape:", forecast_df.shape)

## 9. Chia train/validation/test theo thời gian

In [ ]:
model_df = forecast_df.dropna(subset=["target_revenue_next_30d"]).copy()
feature_cols = [
    c for c in model_df.columns
    if c.startswith(("revenue_lag_", "revenue_mean_", "revenue_std_", "views_lag_", "views_mean_", "purchases_lag_", "purchases_mean_"))
] + ["day_of_week", "month", "day_of_year"]

X_all = pd.get_dummies(model_df[feature_cols + ["country"]], columns=["country"], dtype=float)
y_all = model_df["target_revenue_next_30d"].astype(float)
dates = model_df["date"]
unique_dates = np.array(sorted(dates.unique()))
if len(unique_dates) < 20:
    raise ValueError("Not enough unique dates for reliable chronological train/validation/test split.")

train_cut = unique_dates[int(len(unique_dates) * 0.70)]
val_cut = unique_dates[int(len(unique_dates) * 0.85)]
train_mask = dates <= train_cut
val_mask = (dates > train_cut) & (dates <= val_cut)
test_mask = dates > val_cut

X_train, y_train = X_all.loc[train_mask], y_all.loc[train_mask]
X_val, y_val = X_all.loc[val_mask], y_all.loc[val_mask]
X_test, y_test = X_all.loc[test_mask], y_all.loc[test_mask]

print("Train:", dates[train_mask].min(), "to", dates[train_mask].max(), len(X_train))
print("Validation:", dates[val_mask].min(), "to", dates[val_mask].max(), len(X_val))
print("Test:", dates[test_mask].min(), "to", dates[test_mask].max(), len(X_test))

## 10. Huấn luyện và so sánh baseline, Linear Regression, Random Forest

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

models = {
    "DummyMeanBaseline": DummyRegressor(strategy="mean"),
    "LinearRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LinearRegression()),
    ]),
    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=250, min_samples_leaf=2,
            random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
}
fitted_models, validation_rows = {}, []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    validation_rows.append({"model": name, **regression_metrics(y_val, pred)})
    fitted_models[name] = model

validation_results = pd.DataFrame(validation_rows).sort_values("MAE")
display(validation_results)
best_name = validation_results.iloc[0]["model"]
print("Selected on validation MAE:", best_name)

## 11. Đánh giá test và trực quan hóa model so với baseline

In [ ]:
test_rows, test_predictions = [], {}
for name, model in fitted_models.items():
    pred = model.predict(X_test)
    test_predictions[name] = pred
    test_rows.append({"model": name, **regression_metrics(y_test, pred)})

test_results = pd.DataFrame(test_rows).sort_values("MAE")
display(test_results)
test_results.to_csv(REPORT_DIR / "model_comparison.csv", index=False)

plt.figure(figsize=(9, 4))
plt.bar(test_results["model"], test_results["MAE"])
plt.xticks(rotation=20, ha="right")
plt.ylabel("MAE (lower is better)")
plt.title("Test-set MAE: models vs baseline")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "model_vs_baseline_mae.png", dpi=150)
plt.show()

best_pred = test_predictions[best_name]
comparison = model_df.loc[test_mask, ["date", "country"]].copy()
comparison["actual_revenue_next_30d"] = y_test.to_numpy()
comparison["predicted_revenue_next_30d"] = best_pred
comparison.to_csv(REPORT_DIR / "test_predictions.csv", index=False)

## 12. Biểu đồ dự đoán và thực tế

In [ ]:
chosen_country = comparison["country"].value_counts().index[0]
plot_df = comparison[comparison["country"] == chosen_country].sort_values("date")

plt.figure(figsize=(12, 5))
plt.plot(plot_df["date"], plot_df["actual_revenue_next_30d"], label="Actual")
plt.plot(plot_df["date"], plot_df["predicted_revenue_next_30d"], label="Predicted")
plt.title(f"30-day revenue forecast — {chosen_country}")
plt.xlabel("Forecast origin date")
plt.ylabel("Revenue over next 30 days")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "actual_vs_predicted.png", dpi=150)
plt.show()

## 13. Lưu mô hình và metadata

In [ ]:
train_val_mask = dates <= val_cut
final_model = models[best_name]
final_model.fit(X_all.loc[train_val_mask], y_all.loc[train_val_mask])

artifact = {
    "model": final_model,
    "model_name": best_name,
    "feature_columns": list(X_all.columns),
    "horizon_days": HORIZON_DAYS,
    "target": "target_revenue_next_30d",
    "version": "1.0.0",
}
joblib.dump(artifact, ARTIFACT_DIR / "model.joblib")

metadata = {
    "model_name": best_name,
    "feature_columns": list(X_all.columns),
    "horizon_days": HORIZON_DAYS,
    "target": "target_revenue_next_30d",
    "version": "1.0.0",
    "test_metrics": test_results.to_dict(orient="records"),
}
with (ARTIFACT_DIR / "metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved model and metadata to", ARTIFACT_DIR.resolve())

## 14. Monitoring offline — actual vs predicted

In [ ]:
monitoring = comparison.copy()
monitoring["absolute_error"] = (
    monitoring["actual_revenue_next_30d"] -
    monitoring["predicted_revenue_next_30d"]
).abs()
monitoring["model_version"] = "1.0.0"
monitoring["evaluated_at"] = pd.Timestamp.utcnow()
monitoring.to_csv(REPORT_DIR / "monitoring_results.csv", index=False)
display(monitoring.head())
print("Monitoring MAE:", mean_absolute_error(
    monitoring["actual_revenue_next_30d"],
    monitoring["predicted_revenue_next_30d"]
))

## 15. Tạo skeleton Flask API, unit tests và Docker

In [ ]:
# Generate starter files for deployment. Review before using in production.
project_root = PROJECT_DIR
src_dir = project_root / "src"
tests_dir = project_root / "tests"
for folder in (src_dir, tests_dir):
    folder.mkdir(parents=True, exist_ok=True)
(src_dir / "__init__.py").write_text("", encoding="utf-8")

api_code = r"""
from datetime import date
from pathlib import Path
import logging, time, uuid, json
import joblib
import pandas as pd
from flask import Flask, request, jsonify

app = Flask(__name__)
logging.basicConfig(level=logging.INFO)
ARTIFACT_PATH = Path("artifacts/model.joblib")
DATA_PATH = Path("data/processed/daily_country_revenue.csv")
ARTIFACT = joblib.load(ARTIFACT_PATH) if ARTIFACT_PATH.exists() else None

def available_countries():
    if not DATA_PATH.exists():
        return []
    return sorted(pd.read_csv(DATA_PATH)["country"].dropna().unique().tolist())

def make_features(country, target_date):
    # TODO: implement the exact past-only feature creation used in training.
    # Return a one-row DataFrame with the same feature columns as the saved artifact.
    raise NotImplementedError("Implement production feature generation consistently with training.")

def predict_one(country, target_date):
    if ARTIFACT is None:
        raise RuntimeError("Model artifact not found")
    X = make_features(country, target_date)
    X = X.reindex(columns=ARTIFACT["feature_columns"], fill_value=0)
    return float(ARTIFACT["model"].predict(X)[0])

@app.get("/health")
def health():
    return jsonify({"status": "ok", "model_loaded": ARTIFACT is not None}), 200

@app.post("/predict")
def predict():
    started = time.perf_counter()
    request_id = str(uuid.uuid4())
    body = request.get_json(silent=True)
    if not isinstance(body, dict):
        return jsonify({"error": "JSON body required"}), 400
    country, date_text = body.get("country"), body.get("date")
    if not isinstance(country, str) or not country.strip():
        return jsonify({"error": "country is required"}), 400
    try:
        target_date = date.fromisoformat(date_text)
    except (TypeError, ValueError):
        return jsonify({"error": "date must be YYYY-MM-DD"}), 400
    try:
        countries = available_countries()
        if country.upper() == "ALL":
            result = {"date": date_text, "predictions": {
                c: predict_one(c, target_date) for c in countries
            }}
        elif country not in countries:
            return jsonify({"error": "unknown country"}), 400
        else:
            result = {"country": country, "date": date_text,
                      "predicted_revenue_next_30d": predict_one(country, target_date)}
        app.logger.info(json.dumps({
            "request_id": request_id,
            "timestamp": pd.Timestamp.utcnow().isoformat(),
            "country": country,
            "target_date": date_text,
            "model_version": ARTIFACT.get("version") if ARTIFACT else None,
            "runtime_ms": round((time.perf_counter()-started)*1000, 3),
            "status": "success"
        }))
        result["request_id"] = request_id
        return jsonify(result), 200
    except NotImplementedError as exc:
        return jsonify({"error": str(exc), "request_id": request_id}), 501
    except Exception:
        app.logger.exception("Prediction failed")
        return jsonify({"error": "prediction failed", "request_id": request_id}), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
"""
(src_dir / "api.py").write_text(api_code.strip() + "\n", encoding="utf-8")

test_code = """
from src.api import app

def test_health():
    client = app.test_client()
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json["status"] == "ok"

def test_missing_json_rejected():
    client = app.test_client()
    response = client.post("/predict", data="not json", content_type="text/plain")
    assert response.status_code == 400

def test_invalid_date_rejected():
    client = app.test_client()
    response = client.post("/predict", json={"country": "USA", "date": "bad-date"})
    assert response.status_code == 400
"""
(tests_dir / "test_api.py").write_text(test_code.strip() + "\n", encoding="utf-8")
(project_root / "requirements.txt").write_text(
    "pandas\nnumpy\nscikit-learn\nmatplotlib\nflask\npytest\njoblib\n",
    encoding="utf-8"
)
(project_root / "Dockerfile").write_text(
    "FROM python:3.12-slim\nWORKDIR /app\n"
    "ENV PYTHONDONTWRITEBYTECODE=1\nENV PYTHONUNBUFFERED=1\n"
    "COPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\n"
    "COPY src/ src/\nCOPY artifacts/ artifacts/\nCOPY data/processed/ data/processed/\n"
    "EXPOSE 5000\nCMD [\"python\", \"-m\", \"src.api\"]\n",
    encoding="utf-8"
)
(project_root / ".dockerignore").write_text(
    ".venv/\n__pycache__/\n.pytest_cache/\n.git/\n", encoding="utf-8"
)
print("Generated starter API/tests/Docker files under:", project_root.resolve())
print("IMPORTANT: API feature generation is a TODO; finish it before deployment.")

## 16. Chạy unit tests

In [ ]:
# Run this after generated files and dependencies are available.
import subprocess, sys
test_dir = PROJECT_DIR / "tests"
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(test_dir)],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)
print("pytest exit code:", result.returncode)

## 17. Checklist 12 tiêu chí peer review

In [ ]:
criteria = [
    ("Unit tests for API", "tests/test_api.py: valid/invalid requests"),
    ("Unit tests for model", "Add tests for model loading, feature schema and finite prediction"),
    ("Unit tests for logging", "Add tests using temporary log paths"),
    ("All unit tests run with one script", "Create scripts/run_all_tests.py and run pytest"),
    ("Performance monitoring", "Monitoring table, MAE/RMSE and trend plot"),
    ("Isolate test reads/writes from production", "Use fixtures, tmp_path and mocks"),
    ("API supports one/all countries", "Test country-specific and ALL requests after feature function is implemented"),
    ("Automated data ingestion", "Reusable function and script"),
    ("Multiple models compared", "Baseline + Linear Regression + Random Forest"),
    ("EDA visualizations", "Saved EDA figures"),
    ("Working Docker image", "Build and run Docker image; verify /health"),
    ("Visualization against baseline", "model_vs_baseline_mae.png and actual_vs_predicted.png"),
]
display(pd.DataFrame(criteria, columns=["Criterion", "Evidence / action"]))

## 18. Lệnh chạy và lưu bằng chứng

Sau khi hoàn thiện:
1. Chạy các cell từ trên xuống dưới.
2. Lưu các file trong `aavail_capstone_output/reports/`.
3. Hoàn thiện `make_features()` trong `src/api.py` sao cho dùng đúng feature engineering lúc train.
4. Bổ sung unit tests cho model và logging; tách test I/O khỏi artifacts/logs production.
5. Chạy tests: `python -m pytest -q`.
6. Build/run Docker từ thư mục output sau khi đã có model artifact.
7. Thử `GET /health`, `POST /predict` cho một quốc gia và `POST /predict` với `country="ALL"`.
8. Viết README mô tả dữ liệu, EDA, mô hình, metric thật, API, monitoring, tests và Docker.

**Caveat:** Do không có JSON gốc trong notebook, cần xác nhận ánh xạ cột và cách diễn giải revenue. Cũng cần xác minh việc coi ngày không có bản ghi là doanh thu 0 có phù hợp với dữ liệu hay không. Không nộp khi API feature generation còn TODO hoặc khi tests/Docker chưa thực sự chạy thành công.